# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BarnaliBhowmik/FlyRank_ML/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [1]:
%pip -q install duckdb huggingface_hub


In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [4]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [5]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [6]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [7]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.547     0.335     0.415      9389
           1      0.685     0.839     0.754     16162

    accuracy                          0.654     25551
   macro avg      0.616     0.587     0.585     25551
weighted avg      0.634     0.654     0.630     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [8]:
!pip install -q huggingface_hub duckdb

In [9]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    allow_patterns=[
        "dim_*.parquet",
        "fact_content_query_*.parquet",
        "fact_content_daily_performance/month=2026-07*/**/*.parquet"
    ]
)

print("Downloaded to:", path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


In [10]:
import os

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".parquet"):
            print(os.path.join(root, file))

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_content.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_query_90d.parquet


In [11]:
import duckdb

con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE VIEW dim_clients AS
    SELECT *
    FROM read_parquet('{path}/dim_clients.parquet');
""")

In [12]:
con.execute(f"""
    CREATE OR REPLACE VIEW fact_content_query AS
    SELECT *
    FROM read_parquet('{path}/fact_content_query_*.parquet');
""")

In [14]:
import os

for root, dirs, files in os.walk(path):
    parquet_files = [f for f in files if f.endswith(".parquet")]
    if parquet_files:
        print("\n📁", root)
        for f in parquet_files[:10]:
            print("   ", f)


📁 /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2
    dim_clients.parquet
    dim_content.parquet
    fact_content_query_90d.parquet


In [15]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [16]:
daily_files = [
    f for f in files
    if "fact_content_daily_performance" in f
]

print("Number of daily-performance files:", len(daily_files))

for f in daily_files[:30]:
    print(f)

Number of daily-performance files: 19
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fa

In [17]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    allow_patterns=[
        "dim_clients.parquet",
        "dim_content.parquet",
        "fact_content_query_90d.parquet",
        "fact_content_daily_performance/month=2026-04/*",
        "fact_content_daily_performance/month=2026-05/*",
        "fact_content_daily_performance/month=2026-06/*",
    ]
)

print(path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


In [18]:
import os

for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith(".parquet"):
            print(os.path.relpath(os.path.join(root, f), path))

dim_clients.parquet
dim_content.parquet
fact_content_query_90d.parquet
fact_content_daily_performance/month=2026-05/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=2026-06/data_0.parquet


In [19]:
import duckdb

con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE VIEW fact_daily AS
    SELECT *
    FROM read_parquet(
        '{path}/fact_content_daily_performance/month=*/data_0.parquet',
        hive_partitioning=true
    )
""")

In [20]:
con.sql("""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM fact_daily
""").show()

┌──────────┬────────────┬────────────┐
│   rows   │  min_date  │  max_date  │
│  int64   │    date    │    date    │
├──────────┼────────────┼────────────┤
│ 33806178 │ 2026-04-01 │ 2026-06-30 │
└──────────┴────────────┴────────────┘



In [21]:
con.sql("DESCRIBE fact_daily").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [22]:
con.sql("""
    SELECT *
    FROM fact_daily
    LIMIT 5
""").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

In [24]:
features = con.sql("""
    SELECT
        client_hash_id,

        -- =========================
        -- Google Search Console
        -- =========================
        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS gsc_impressions_30d,

        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS gsc_clicks_30d,

        AVG(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                THEN gsc_avg_position
            END
        ) AS gsc_avg_position_30d,

        -- =========================
        -- Sessions
        -- =========================
        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(sessions_organic, 0)
                ELSE 0
            END
        ) AS organic_sessions_30d,

        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(sessions_direct, 0)
                ELSE 0
            END
        ) AS direct_sessions_30d,

        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(sessions_referral, 0)
                ELSE 0
            END
        ) AS referral_sessions_30d,

        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(sessions_social, 0)
                ELSE 0
            END
        ) AS social_sessions_30d,

        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(sessions_paid, 0)
                ELSE 0
            END
        ) AS paid_sessions_30d,

        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(sessions_ai, 0)
                ELSE 0
            END
        ) AS ai_sessions_30d,

        -- =========================
        -- AI crawler activity
        -- =========================
        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(ai_chatgpt, 0)
                ELSE 0
            END
        ) AS chatgpt_30d,

        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(ai_perplexity, 0)
                ELSE 0
            END
        ) AS perplexity_30d,

        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(ai_gemini, 0)
                ELSE 0
            END
        ) AS gemini_30d,

        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(ai_copilot, 0)
                ELSE 0
            END
        ) AS copilot_30d,

        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(ai_claude, 0)
                ELSE 0
            END
        ) AS claude_30d,

        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(ai_meta, 0)
                ELSE 0
            END
        ) AS meta_ai_30d,

        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(ai_other, 0)
                ELSE 0
            END
        ) AS other_ai_30d,

        -- =========================
        -- Engagement
        -- =========================
        SUM(
            CASE
                WHEN report_date >= (SELECT MAX(report_date) FROM fact_daily)
                     - INTERVAL 30 DAY
                THEN COALESCE(scroll_events, 0)
                ELSE 0
            END
        ) AS scroll_events_30d

    FROM fact_daily

    GROUP BY client_hash_id
""").df()

print("Feature table shape:", features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature table shape: (66, 18)


,client_hash_id,gsc_impressions_30d,gsc_clicks_30d,gsc_avg_position_30d,organic_sessions_30d,direct_sessions_30d,referral_sessions_30d,social_sessions_30d,paid_sessions_30d,ai_sessions_30d,chatgpt_30d,perplexity_30d,gemini_30d,copilot_30d,claude_30d,meta_ai_30d,other_ai_30d,scroll_events_30d
0,client_62f4a7e64f5e0096,26586696.0,98018.0,23.070546,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,client_9958f0a7ae1df715,388263.0,622.0,51.197721,780.0,2533.0,355.0,14.0,0.0,76.0,53.0,1.0,4.0,13.0,5.0,0.0,0.0,3630.0
2,client_73cda7b4e4f265ea,59940381.0,184030.0,24.255168,251259.0,176877.0,65114.0,31984.0,246084.0,3317.0,2795.0,144.0,293.0,12.0,83.0,0.0,0.0,63178.0
3,client_f623b01661d4bfe4,81480.0,1041.0,NaN,1067.0,314.0,22.0,23.0,340.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,297.0
4,client_8ae2bfb5aa1ffa1e,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
con.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients
    FROM fact_daily
""").show()

┌──────────┬─────────┐
│   rows   │ clients │
│  int64   │  int64  │
├──────────┼─────────┤
│ 33806178 │      66 │
└──────────┴─────────┘



In [28]:
print(features.shape)
features.head()

(66, 18)


,client_hash_id,gsc_impressions_30d,gsc_clicks_30d,gsc_avg_position_30d,organic_sessions_30d,direct_sessions_30d,referral_sessions_30d,social_sessions_30d,paid_sessions_30d,ai_sessions_30d,chatgpt_30d,perplexity_30d,gemini_30d,copilot_30d,claude_30d,meta_ai_30d,other_ai_30d,scroll_events_30d
0,client_62f4a7e64f5e0096,26586696.0,98018.0,23.070546,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,client_9958f0a7ae1df715,388263.0,622.0,51.197721,780.0,2533.0,355.0,14.0,0.0,76.0,53.0,1.0,4.0,13.0,5.0,0.0,0.0,3630.0
2,client_73cda7b4e4f265ea,59940381.0,184030.0,24.255168,251259.0,176877.0,65114.0,31984.0,246084.0,3317.0,2795.0,144.0,293.0,12.0,83.0,0.0,0.0,63178.0
3,client_f623b01661d4bfe4,81480.0,1041.0,NaN,1067.0,314.0,22.0,23.0,340.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,297.0
4,client_8ae2bfb5aa1ffa1e,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
con.execute(f"""
    CREATE OR REPLACE VIEW fact_query AS
    SELECT *
    FROM read_parquet('{path}/fact_content_query_90d.parquet')
""")

In [30]:
con.sql("DESCRIBE fact_query").show()

┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_hash_id                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_char_count              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ query_token_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ window_start                  │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ window_end                    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ impressions_90d               

In [32]:
query_features = con.sql("""
    SELECT
        client_hash_id,

        -- Query volume / diversity
        COUNT(DISTINCT query_hash_id) AS unique_queries_90d,
        COUNT(DISTINCT content_hash_id) AS unique_contents_90d,

        -- Query characteristics
        AVG(query_char_count) AS avg_query_chars,
        AVG(query_token_count) AS avg_query_tokens,

        -- Overall search performance
        SUM(impressions_90d) AS impressions_90d,
        SUM(clicks_90d) AS clicks_90d,

        -- Recent vs previous period
        SUM(impressions_last30) AS impressions_last30,
        SUM(clicks_last30) AS clicks_last30,
        SUM(impressions_prev30) AS impressions_prev30,
        SUM(clicks_prev30) AS clicks_prev30,

        -- Search position
        AVG(avg_position_90d) AS avg_position_90d,
        AVG(avg_position_last30) AS avg_position_last30,
        AVG(avg_position_prev30) AS avg_position_prev30,

        -- Query visibility / concentration
        AVG(content_total_impressions_90d)
            AS avg_content_impressions_90d,

        AVG(content_visible_query_count)
            AS avg_visible_query_count,

        -- Long-tail / rare-query signals
        SUM(rare_query_count)
            AS rare_queries_90d,

        AVG(rare_impressions_share)
            AS avg_rare_impressions_share,

        AVG(anonymized_impressions_share)
            AS avg_anonymized_impressions_share

    FROM fact_query

    GROUP BY client_hash_id
""").df()

print("Query feature shape:", query_features.shape)
query_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query feature shape: (52, 19)


,client_hash_id,unique_queries_90d,unique_contents_90d,avg_query_chars,avg_query_tokens,impressions_90d,clicks_90d,impressions_last30,clicks_last30,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,avg_content_impressions_90d,avg_visible_query_count,rare_queries_90d,avg_rare_impressions_share,avg_anonymized_impressions_share
0,client_08a6a72ff48e62c0,45336,13455,30.492605,5.280759,6545917.0,16478.0,1702559.0,2544.0,2432725.0,6783.0,32.810049,39.842164,34.698682,15124.930789,39.801124,59794692.0,0.157081,0.482091
1,client_62f4a7e64f5e0096,139833,18161,34.546102,5.850368,26650172.0,44756.0,6195327.0,10992.0,9280876.0,16597.0,15.049940,22.382884,19.021630,32396.819093,98.231425,150409896.0,0.085393,0.619707
2,client_73cda7b4e4f265ea,313829,26110,34.164795,5.519629,68026278.0,137189.0,18093647.0,34028.0,24856443.0,51886.0,15.459150,21.104118,17.730776,42663.010609,107.480014,364999154.0,0.076893,0.601136
3,client_7de9989c909e91a5,147,71,30.739362,5.436170,12436.0,18.0,11939.0,15.0,497.0,3.0,17.921285,18.567627,19.540232,1284.212766,5.031915,8413.0,0.140499,0.604694
4,client_9958f0a7ae1df715,7584,1926,23.597207,4.324123,380914.0,218.0,134173.0,67.0,125794.0,67.0,53.908909,53.531090,54.609112,1169.628353,11.845976,631259.0,0.276021,0.289485


In [33]:
features_final = features.merge(
    query_features,
    on="client_hash_id",
    how="left"
)

print("Final feature shape:", features_final.shape)
features_final.head()

Final feature shape: (66, 36)


,client_hash_id,gsc_impressions_30d,gsc_clicks_30d,gsc_avg_position_30d,organic_sessions_30d,direct_sessions_30d,referral_sessions_30d,social_sessions_30d,paid_sessions_30d,ai_sessions_30d,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,avg_content_impressions_90d,avg_visible_query_count,rare_queries_90d,avg_rare_impressions_share,avg_anonymized_impressions_share
0,client_62f4a7e64f5e0096,26586696.0,98018.0,23.070546,0.0,0.0,0.0,0.0,0.0,0.0,...,9280876.0,16597.0,15.049940,22.382884,19.021630,32396.819093,98.231425,150409896.0,0.085393,0.619707
1,client_9958f0a7ae1df715,388263.0,622.0,51.197721,780.0,2533.0,355.0,14.0,0.0,76.0,...,125794.0,67.0,53.908909,53.531090,54.609112,1169.628353,11.845976,631259.0,0.276021,0.289485
2,client_73cda7b4e4f265ea,59940381.0,184030.0,24.255168,251259.0,176877.0,65114.0,31984.0,246084.0,3317.0,...,24856443.0,51886.0,15.459150,21.104118,17.730776,42663.010609,107.480014,364999154.0,0.076893,0.601136
3,client_f623b01661d4bfe4,81480.0,1041.0,NaN,1067.0,314.0,22.0,23.0,340.0,3.0,...,72334.0,598.0,61.003776,58.502570,60.276744,2456.483718,30.613693,95401.0,0.108401,0.073626
4,client_8ae2bfb5aa1ffa1e,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,135.0,0.0,22.150946,NaN,23.635793,216.000000,2.230769,223.0,0.203579,0.394748


In [34]:
print(
    features_final.isna().sum()
    .sort_values(ascending=False)
    .head(15)
)

gsc_avg_position_30d                18
avg_position_last30                 17
unique_contents_90d                 14
rare_queries_90d                    14
avg_rare_impressions_share          14
avg_anonymized_impressions_share    14
clicks_last30                       14
unique_queries_90d                  14
avg_content_impressions_90d         14
avg_position_prev30                 14
avg_position_90d                    14
avg_visible_query_count             14
clicks_90d                          14
impressions_prev30                  14
avg_query_chars                     14
dtype: int64


In [35]:
print(features_final.shape)

(66, 36)


In [38]:
labels = con.sql("""
    WITH monthly AS (
        SELECT
            client_hash_id,
            DATE_TRUNC('month', report_date) AS month,
            SUM(COALESCE(gsc_impressions, 0)) AS impressions
        FROM fact_daily
        GROUP BY
            client_hash_id,
            DATE_TRUNC('month', report_date)
    ),

    pivoted AS (
        SELECT
            client_hash_id,

            SUM(
                CASE
                    WHEN month = DATE '2026-05-01'
                    THEN impressions
                    ELSE 0
                END
            ) AS may_impressions,

            SUM(
                CASE
                    WHEN month = DATE '2026-06-01'
                    THEN impressions
                    ELSE 0
                END
            ) AS june_impressions

        FROM monthly
        GROUP BY client_hash_id
    )

    SELECT
        client_hash_id,
        may_impressions,
        june_impressions,

        CASE
            WHEN may_impressions > 0
             AND june_impressions < may_impressions * 0.80
            THEN 1
            ELSE 0
        END AS declining

    FROM pivoted
""").df()

print(labels.shape)
labels.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(66, 4)


,client_hash_id,may_impressions,june_impressions,declining
0,client_62f4a7e64f5e0096,34041513.0,25367164.0,1
1,client_4a18d1793d92fb84,7876.0,4249.0,1
2,client_3ffa76342f366962,80005.0,49718.0,1
3,client_08a6a72ff48e62c0,13455622.0,5950520.0,1
4,client_f623b01661d4bfe4,94880.0,78257.0,0


In [39]:
print(labels["declining"].value_counts())

declining
0    37
1    29
Name: count, dtype: int64


In [40]:
model_data = features_final.merge(
    labels,
    on="client_hash_id",
    how="inner"
)

print("Model data shape:", model_data.shape)

Model data shape: (66, 39)


In [41]:
print(model_data["declining"].value_counts())

declining
0    37
1    29
Name: count, dtype: int64


In [42]:
features_may = con.sql("""
    SELECT
        client_hash_id,

        SUM(COALESCE(gsc_impressions, 0)) AS gsc_impressions_may,
        SUM(COALESCE(gsc_clicks, 0)) AS gsc_clicks_may,

        AVG(gsc_avg_position) AS gsc_avg_position_may,

        SUM(COALESCE(sessions_organic, 0)) AS organic_sessions_may,
        SUM(COALESCE(sessions_direct, 0)) AS direct_sessions_may,
        SUM(COALESCE(sessions_referral, 0)) AS referral_sessions_may,
        SUM(COALESCE(sessions_social, 0)) AS social_sessions_may,
        SUM(COALESCE(sessions_paid, 0)) AS paid_sessions_may,
        SUM(COALESCE(sessions_ai, 0)) AS ai_sessions_may,

        SUM(COALESCE(ai_chatgpt, 0)) AS chatgpt_may,
        SUM(COALESCE(ai_perplexity, 0)) AS perplexity_may,
        SUM(COALESCE(ai_gemini, 0)) AS gemini_may,
        SUM(COALESCE(ai_copilot, 0)) AS copilot_may,
        SUM(COALESCE(ai_claude, 0)) AS claude_may,
        SUM(COALESCE(ai_meta, 0)) AS meta_ai_may,
        SUM(COALESCE(ai_other, 0)) AS other_ai_may,

        SUM(COALESCE(scroll_events, 0)) AS scroll_events_may

    FROM fact_daily

    WHERE report_date >= DATE '2026-05-01'
      AND report_date <  DATE '2026-06-01'

    GROUP BY client_hash_id
""").df()

print("May features:", features_may.shape)
features_may.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

May features: (66, 18)


,client_hash_id,gsc_impressions_may,gsc_clicks_may,gsc_avg_position_may,organic_sessions_may,direct_sessions_may,referral_sessions_may,social_sessions_may,paid_sessions_may,ai_sessions_may,chatgpt_may,perplexity_may,gemini_may,copilot_may,claude_may,meta_ai_may,other_ai_may,scroll_events_may
0,client_3ffa76342f366962,80005.0,1115.0,4.819905,1839.0,678.0,70.0,16.0,0.0,70.0,38.0,24.0,0.0,8.0,0.0,0.0,0.0,930.0
1,client_62f4a7e64f5e0096,34041513.0,90269.0,19.343221,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,client_9958f0a7ae1df715,387477.0,538.0,47.568549,726.0,1634.0,275.0,1.0,0.0,35.0,27.0,1.0,1.0,3.0,3.0,0.0,0.0,2710.0
3,client_73cda7b4e4f265ea,80876427.0,242611.0,18.580102,298449.0,42253.0,1642.0,268.0,73116.0,561.0,212.0,207.0,77.0,4.0,61.0,0.0,0.0,45855.0
4,client_fef1a8f436438636,7467217.0,21570.0,25.812403,26218.0,7173.0,525.0,41.0,44.0,98.0,60.0,22.0,10.0,6.0,0.0,0.0,0.0,3040.0


In [43]:
query_features_may = con.sql("""
    SELECT
        client_hash_id,

        COUNT(DISTINCT query_hash_id) AS unique_queries_may,
        COUNT(DISTINCT content_hash_id) AS unique_contents_may,

        AVG(query_char_count) AS avg_query_chars,
        AVG(query_token_count) AS avg_query_tokens,

        SUM(impressions_last30) AS impressions_may,
        SUM(clicks_last30) AS clicks_may,

        AVG(avg_position_last30) AS avg_position_may,

        AVG(content_visible_query_count) AS avg_visible_query_count,

        SUM(rare_query_count) AS rare_queries,
        AVG(rare_impressions_share) AS rare_impressions_share,
        AVG(anonymized_impressions_share) AS anonymized_impressions_share

    FROM fact_query

    WHERE window_end >= DATE '2026-05-01'
      AND window_end <  DATE '2026-06-01'

    GROUP BY client_hash_id
""").df()

print("May query features:", query_features_may.shape)
query_features_may.head()

May query features: (0, 12)


,client_hash_id,unique_queries_may,unique_contents_may,avg_query_chars,avg_query_tokens,impressions_may,clicks_may,avg_position_may,avg_visible_query_count,rare_queries,rare_impressions_share,anonymized_impressions_share


In [44]:
features_may_final = features_may.merge(
    query_features_may,
    on="client_hash_id",
    how="left"
)

print("Final May feature shape:", features_may_final.shape)

Final May feature shape: (66, 29)


In [45]:
labels = con.sql("""
    WITH monthly AS (
        SELECT
            client_hash_id,
            DATE_TRUNC('month', report_date) AS month,
            SUM(COALESCE(gsc_impressions, 0)) AS impressions
        FROM fact_daily
        GROUP BY client_hash_id, DATE_TRUNC('month', report_date)
    )

    SELECT
        client_hash_id,

        MAX(CASE
            WHEN month = DATE '2026-05-01'
            THEN impressions
        END) AS may_impressions,

        MAX(CASE
            WHEN month = DATE '2026-06-01'
            THEN impressions
        END) AS june_impressions,

        CASE
            WHEN MAX(CASE
                WHEN month = DATE '2026-05-01'
                THEN impressions
            END) > 0
            AND
            MAX(CASE
                WHEN month = DATE '2026-06-01'
                THEN impressions
            END)
            <
            0.80 * MAX(CASE
                WHEN month = DATE '2026-05-01'
                THEN impressions
            END)
            THEN 1
            ELSE 0
        END AS declining

    FROM monthly
    GROUP BY client_hash_id
""").df()

print(labels["declining"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

declining
0    38
1    28
Name: count, dtype: int64


In [46]:
model_data = features_may_final.merge(
    labels,
    on="client_hash_id",
    how="inner"
)

print("Model data:", model_data.shape)
print(model_data["declining"].value_counts())

Model data: (66, 32)
declining
0    38
1    28
Name: count, dtype: int64


In [47]:
import numpy as np
import pandas as pd

# Separate IDs and target
X = model_data.drop(
    columns=["client_hash_id", "may_impressions", "june_impressions", "declining"]
)

y = model_data["declining"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

X shape: (66, 28)
y shape: (66,)

Target distribution:
declining
0    38
1    28
Name: count, dtype: int64


In [48]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_imputed = imputer.fit_transform(X)

print("Missing values after imputation:",
      np.isnan(X_imputed).sum())

Missing values after imputation: 0


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['unique_queries_may' 'unique_contents_may' 'avg_query_chars'
 'avg_query_tokens' 'impressions_may' 'clicks_may' 'avg_position_may'
 'avg_visible_query_count' 'rare_queries' 'rare_impressions_share'
 'anonymized_impressions_share']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [49]:
from sklearn.model_selection import GroupShuffleSplit

groups = model_data["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X_imputed, y, groups=groups)
)

X_train = X_imputed[train_idx]
X_test = X_imputed[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training samples:", len(train_idx))
print("Test samples:", len(test_idx))

Training samples: 52
Test samples: 14


In [50]:
train_clients = set(
    model_data.iloc[train_idx]["client_hash_id"]
)

test_clients = set(
    model_data.iloc[test_idx]["client_hash_id"]
)

print("Overlapping clients:",
      len(train_clients & test_clients))

Overlapping clients: 0


In [51]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.70      0.78      0.74         9
           1       0.50      0.40      0.44         5

    accuracy                           0.64        14
   macro avg       0.60      0.59      0.59        14
weighted avg       0.63      0.64      0.63        14



In [52]:
print("Train label distribution:")
print(y_train.value_counts())

print("\nTest label distribution:")
print(y_test.value_counts())

Train label distribution:
declining
0    29
1    23
Name: count, dtype: int64

Test label distribution:
declining
0    9
1    5
Name: count, dtype: int64
